# OpenDC Demo 1
### First experiment

Datacenters are becoming an increasingly large contributor to the global carbon footprint. However, because of a lack of tools/guidelines, it has been challenging to optimize datacenters for carbon emissions. This is amplified by the fact that running experiments on datacenters is both expensive and time-consuming. 

OpenDC is an event-based discrete datacenter simulator. Using such a tool, we can do experiments on datacenters in a cost-effective and flexible way. In this demo, we will learn how to conduct a simple experiment. After, we learn how to aggregate and visualize the results for more insights.  

You can read more about OpenDC [here](https://opendc.org/) and [here](https://opendc.org/learn/intro).

# Topology

To run a simulation, OpenDC needs a definition of the datacenter, which we call the topology. The topology of a datacenter can influence its performance and sustainability greatly. It determines which tasks can be run, how efficiently they are run, and if they can be executed in parallel.

The topology of a data center is provided using a JSON file. This file defines the number of clusters available, the hosts they contain, and the type of hosts they comprise. The topology file used for the Surf workload is shown below and can be found [here](topologies/1.first_experiment/surfsara.json):

```json
{
    "clusters":
    [
        {
            "name": "C01",
            "hosts" :
            [
                {
                    "name": "H01",
                    "cpu":
                    {
                        "coreCount": 16,
                        "coreSpeed": 2100
                    },
                    "memory": {
                        "memorySize": 100000
                    },
                    "cpuPowerModel": {
                        "modelType": "linear",
                        "power": 400.0,
                        "idlePower": 32.0,
                        "maxPower": 180.0
                    },
                    "count": 279
                }
            ],
            "powerSource": {
                "carbonTracePath": "carbon_traces/NL_2021-2024.parquet"
            }
        }
    ]
}
```

The small datacenter contains only a single cluster *C01*, which contains 279 host of type *H01*. Host *H01* constains a single CPU with 16 cores running at 2100 Mhz, and has a memory of 100000 Bytes.

# Workloads

To run a simulation, OpenDC requires information on the type of workload to execute.

The workload is provided using two files:
<ul>
    <li> **tasks.parquet** provides a general overview of the tasks executed during the workload. </li>
    <li> **fragments.parquet** provides detailed information about each task during its runtime. </li>
</ul>

In this demo, we are running the [surf_week](workload_traces/surf_week/) dataset as our workload. 
The surf_week workload is a week-long workload collected from the surf LISA cluster.
The workload consists of 6295 tasks with an average runtime of three hours. 

##### Let's have a look at the files

In [ ]:
import pandas as pd

df_tasks = pd.read_parquet("workload_traces/surf_week/tasks.parquet")
df_fragments = pd.read_parquet("workload_traces/surf_week/fragments.parquet")

In [ ]:
df_tasks.head()

### Fragments

<img src="./figures/fragments.jpg" width=600, alt="Alternative text" />

##### One week of surfsara tasks has over 2 million fragments

In [ ]:
df_fragments.head()

# Carbon Trace

- Carbon Traces define the Carbon Intenisty of the available energy over time

- Collected using [ElectricityMaps](https://portal.electricitymaps.com) and [ENTSO-E](https://www.entsoe.eu/)

- Specific to the location of the datacenter

- Defined as a JSON file

In [ ]:
df_carbon = pd.read_parquet("carbon_traces/NL_2021-2024.parquet")

df_carbon.head()

# Experiment

Finally, OpenDC needs an experiment file. The Experiment file describes what needs to be run, how, and when. An experiment is defined using a JSON file. The experiment we will use for this demo can be found [here](experiments/1.first_experiment/simple_experiment.json) and is shown below:

```json
{
    "name": "1.first_experiment",
    "topologies": [
        {
            "pathToFile": "topologies/1.first_experiment/surfsara.json"
        }
    ],
    "workloads": [
        {
            "pathToFile": "workload_traces/surf_week",
            "type": "ComputeWorkload"
        }
    ],
    "exportModels": [
        {
            "exportInterval": 3600,
            "printFrequency": 1680,
            "filesToExport": [
                "host",
                "powerSource",
                "service",
                "task"
            ]
        }
    ]
}
```

The scenario file used in this demo defines four variables:
- "name" defines where the output files will be stored
- "topologies" defines the different topologies that will be used in the experiments
- "workloads" defines what workloads will be run
- "exportModels" defines how frequently OpenDC should export data


Note: most of the variables in the scenario file are provided as lists. This enables the execution of different experiments within the same scenario. Graph Greenifier will run all combinations of variables as separate experiments. 

# Running an Experiment

An experiment can be run directly from the terminal using the OpenDCExperimentRunner.

In [ ]:
import subprocess

pathToScenario = "experiments/1.first_experiment/simple_experiment.json"
subprocess.run(["OpenDCExperimentRunner/bin/OpenDCExperimentRunner", "--experiment-path", pathToScenario])

## Loading Output Data

OpenDC exports all output metrics to parquet files in the output folder.

You can load the files using Pandas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


df_host = pd.read_parquet("output/1.first_experiment/raw-output/0/seed=0/host.parquet")
df_powerSource = pd.read_parquet("output/1.first_experiment/raw-output/0/seed=0/powerSource.parquet")
df_task = pd.read_parquet("output/1.first_experiment/raw-output/0/seed=0/task.parquet")
df_service = pd.read_parquet("output/1.first_experiment/raw-output/0/seed=0/service.parquet")

### Host
- Information about the host at each timestamp. 
- Examples of metrics: 
    - cpu_utilization
    - power_draw 
    - energy_usage 

In [ ]:
print(f"The host file contains the following columns:\n {np.array(df_host.columns)}\n")
print(f"The host file consist of {len(df_host)} samples")
df_host.head()

### Tasks
- The task file contains all information about the different tasks at each timestamp. 
- Example use cases:
    - when is a task run
    - How long did it take
    - on which host was a task executed

In [ ]:
print(f"The task file contains the following columns:\n {np.array(df_task.columns)}")
print(f"The task file consist of {len(df_task)} samples")
df_task.head()

### Power Source
- The task file contains all information about the power sources at each timestamp. 
- Example use cases:
    - What is the total energy used during the workload?

In [ ]:
print(f"The task file contains the following columns:\n {np.array(df_powerSource.columns)}")
print(f"The power file consist of {len(df_powerSource)} samples")
df_powerSource.head()

### Service

- The service file contains genaral information about the experiments. 
- Example uses:
    - How many tasks are running?
    - How many hosts are up?

In [ ]:
print(f"The service file contains the following columns:\n {np.array(df_service.columns)}")
print(f"The service file consist of {len(df_service)} samples")
df_service.head()

## Aggregating results

- To properly compare the different experiments, we would like to aggregate them into meaningful values.

In [ ]:
runtime = pd.to_timedelta(df_service.timestamp.max() - df_service.timestamp.min(), unit="ms")

print(f"The datacenter finished the workload in {runtime}")

utilization = df_host.cpu_utilization.mean()

print(f"On average, the utilization of each host in the small datacenter is {utilization * 100:.2f}%")

### Sustainability

We can also compare the two datacenters in terms of sustainabilty

Next we print the total energy usage of the two datacenters

In [ ]:
energy = df_powerSource.energy_usage.sum() / 3_600_000 # convert energy to kWh

print(f"The datacenter used {energy:.2f} kWh during the workload")

We can also calculate the carbon emissions of the data center.

**Exercise 1:** 
Calculate the total carbon emission of the datacenter when running the workload.

In [ ]:
# Your code goes here...

<details>
<summary>Click to reveal the answer to Exercise 1</summary>

```python
carbon = df_powerSource.carbon_emission.sum() / 1000  # convert carbon to kg

print(f"The datacenter emitted {carbon:.2f} kg during the workload")
```

</details>

## Visualization

While single numbers can be useful for comparing different workloads, they do not always indicate the reasons for the differences. 

Similarly to value aggregation, vizualization can be done directly using Pandas dataframes. 
However, Graph Greenifier also provides several predefined plotting tools to help this process.

### Active Tasks

Let's start with plotting general information using the service output file. This can be done using the *plotService* function. 
Below, we plot the number of active servers during the workload. 

In [ ]:
plt.plot(df_service.tasks_active)

plt.title("active tasks during a workload")
plt.xlabel("time (h)")
plt.ylabel("active tasks")
plt.show()

### Hosts

We can also look at the performance of the hosts. 

Lets plot the utilization of the hosts over time.

In [ ]:
def plotHost(df_host, column, aggregation_method, window_size=1000):
    if aggregation_method not in ["mean", "sum"]:
        raise ValueError(f"incorrect aggregation method provided: {aggregation_method}, please pick on of [mean, sum]")

    df_agg = df_host.groupby("timestamp")[[column]].agg(aggregation_method)

    plt.plot(df_agg.index/1000/60/60, df_agg.rolling(window_size, min_periods=1).mean())
    plt.xlabel("timestamp (h)")
    plt.ylabel(column)


plotHost(df_host, "cpu_utilization", "mean")

### Sustainability

We can also plot sustainability related metrics.

Lets plot the energy usage over time

In [ ]:
plt.plot(df_powerSource.energy_usage / 3_600_000)

plt.title("Energy usage during a workload")
plt.xlabel("time (h)")
plt.ylabel("energy usage (kWh)")
plt.show()

**Exercise 2:** 

Plot the Carbon Emission over time

In [ ]:
# Your code goes here...

<details>
<summary>Click to reveal the answer to Exercise 2</summary>

```python
plt.plot(df_powerSource.carbon_emission / 1000)

plt.title("Carbon Emission during a Workload")
plt.xlabel("time (h)")
plt.ylabel("carbon emission (kg)")
plt.show()
```

</details>